# Unix-terminal. Работа с файлами


## Мотивация

Терминал отвязывает работу от конкретного компьютера. К серверу можно подключиться с ноутбука, домашнего компьютера и даже с телефона через SSH-клиент вроде Termius — например, проверить процесс или освободить место, пока едешь в метро. Но на маленьком экране особенно важно понимать, где находишься, что удаляешь, кому принадлежат файлы и какие процессы запущены. Команды из этого семинара — базовый язык почти всей дальнейшей работы с Linux.


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Командам, которые мы сегодня разбираем, около полувека: `ls`, `cat`, `chmod` появились в Unix в начале 1970-х и с тех пор почти не изменились. Живучесть у них не из-за традиции, а из-за физики канала: по SSH летят байты текста, а не картинка экрана, поэтому терминал одинаково работает и на локальной машине, и на сервере через мобильный интернет в метро.

Отсюда же привычка администраторов не ставить графику на сервер: GUI на удалённой машине — это лишние гигабайты, лишние процессы и лишние способы что-нибудь сломать. На учебной виртуалке, на кластере с GPU и в docker-контейнере вас в любом случае встретит чёрное окно с приглашением командной строки.

</details>


> **Где работаем.** Все команды выполняются на вашей виртуальной машине, в каталоге `~/seminar-02/`. Каталог `/tmp` для работы не используем: он вычищается при перезагрузке, а результаты нужны и после занятия.
>
> Каждая ячейка `%%bash` запускает **новый** shell: переменные и текущий каталог из предыдущей ячейки в следующей уже не действуют. Поэтому почти каждая ячейка начинается с `cd ~/seminar-02 || exit 1`.
>
> Свёрнутые блоки **🎙 Заметка преподавателя** рассказываются вслух на занятии; при подготовке к защите разворачивайте их сами.


In [ ]:
%%bash
rm -rf ~/seminar-02      # чистый старт: путь написан целиком, без переменных и без «*»
mkdir -p ~/seminar-02    # рабочий каталог семинара — переживёт перезагрузку, в отличие от /tmp
cd ~/seminar-02 || exit 1   # || exit 1: если каталога нет, продолжать ячейку бессмысленно
pwd                      # печатаем полный путь — так видно, где мы оказались на самом деле


## 1. Справка, `echo`, переменные и код завершения


### Справка по командам

Если забыты флаги или порядок аргументов, используют `имя_команды --help`, например `ls --help`. Полная справка открывается через `man имя_команды`: `/текст` ищет, `q` закрывает.

Для встроенных команд Bash используется `help`, например `help echo`. `which name` или `whereis name` показывает путь к найденной команде.


In [ ]:
%%bash
which bash   # путь к файлу программы: видно, что за команду мы на самом деле запускаем
help echo    # echo встроен в сам Bash, поэтому про него рассказывает help, а не man


### `echo`

`echo` выводит аргументы и добавляет перевод строки. `-n` убирает его, `-e` включает спецпоследовательности: `\n` — новая строка, `\t` — табуляция. Другие последовательности перечислены в `help echo`.

В одинарных кавычках текст остаётся буквальным. В двойных кавычках Bash раскрывает переменные.


In [ ]:
%%bash
course_name='Linux practice'

echo "course=$course_name"            # двойные кавычки: Bash подставит значение переменной
echo 'course=$course_name'            # одинарные: на экран уйдёт буквальный текст $course_name
echo -n 'without newline; '           # -n не добавляет перевод строки — следующий вывод будет рядом
echo -e 'first line\n\tsecond line'   # -e включает \n и \t внутри строки


### Переменные

Имя переменной состоит из латинских букв, цифр и `_`, но не начинается с цифры. Дефисы, точки и пробелы в имени запрещены. Значение с пробелами заключают в двойные кавычки. Обычно shell-переменные называют в `snake_case`, переменные окружения — в `UPPER_SNAKE_CASE`.

Фигурные скобки отделяют имя переменной от соседнего текста: `"${group}_report.txt"`. Без скобок `$group_report` означает переменную с именем `group_report`, а не `$group` и суффикс.

`${NAME:-default}` берёт значение `NAME`, если оно задано и не пусто, иначе использует `default`.


In [ ]:
%%bash
student_group_name='ML 01'                          # значение с пробелом — потому и в кавычках
echo "without braces: $student_group_name_report"   # такой переменной нет: подставится пустота
echo "with braces: ${student_group_name}_report"    # скобки отделили имя переменной от суффикса
echo "report=${student_group_name}_report.txt"      # так собирают имя файла из значения переменной
echo "log level=${COURSE_LOG_LEVEL:-info}"          # переменная не задана — подставилось info


#### ❓ **Вопрос**: Почему `$student_group_name_report` вывелось пустым, а не как `ML 01_report`?

<details>

<summary><strong>Ответ</strong></summary>

Bash считает именем переменной максимально длинную допустимую последовательность символов, поэтому он ищет переменную `student_group_name_report`. Такой переменной нет, и подстановка даёт пустую строку. Границу имени задают фигурные скобки: `${student_group_name}_report`.

</details>


### Код завершения

Каждая команда возвращает код: `0` — успех, другое значение — ошибка. `$?` содержит код последней команды. `first && second` выполняет вторую команду после успеха, `first || second` — после ошибки. `true` всегда успешна, `false` всегда завершается ошибкой.


In [ ]:
%%bash
false                         # команда, которая не делает ничего и всегда «падает»
exit_code=$?                  # $? хранит код только последней команды — забираем его сразу
echo "exit_code=$exit_code"   # 1: любой ненулевой код означает ошибку
true && echo 'success'        # && запускает правую часть только после успеха левой
false || echo 'failure'       # || запускает правую часть только после ошибки левой


#### ❓ **Вопрос**: Пусть `group='ML 01'`. Что выведут `echo '$group'`, `echo "$group"` и `false || echo failed`?

<details>

<summary><strong>Ответ</strong></summary>

Первая команда выведет буквальный текст `$group`, вторая — `ML 01`, третья — `failed`, потому что `false` завершилась с ненулевым кодом.

</details>


## 2. stdin, stdout, stderr и перенаправления

Процесс — запущенный экземпляр программы. Терминал запускает процессы команд и связывает их стандартные потоки.

`cat file` читает файл и пишет его содержимое в stdout. Без имени файла `cat` читает stdin.

У процесса есть три стандартных потока:

- stdin (`0`) — входные данные;
- stdout (`1`) — обычный результат;
- stderr (`2`) — ошибки и диагностика.


`>` перезаписывает файл, `>>` дописывает, `<` подаёт файл в stdin. `2>` сохраняет stderr отдельно, `2>&1` направляет stderr туда же, куда уже направлен stdout. Перенаправления обрабатываются слева направо: `> file 2>&1` отправляет оба потока в файл, а `2>&1 > file` перенаправляет туда только stdout — stderr уже получил прежнее назначение stdout. `/dev/null` — специальный файл-приёмник: записанные туда данные отбрасываются.

Обратный слеш `\` в конце строки продолжает ту же команду на следующей строке.

Три потока процесса и перенаправления


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

stdout и stderr независимы и могут буферизоваться по-разному: при выводе в файл stdout обычно копится блоками по несколько килобайт, а stderr пишется сразу. Поэтому после объединения в один файл точный взаимный порядок строк не гарантирован — знакомая боль тех, кто разбирает лог упавшего обучения и видит traceback раньше строки «эпоха 7 началась».

Порядок `> file 2>&1` разбирают на собеседованиях и ломают в реальных cron-задачах: строчка `0 3 * * * backup.sh 2>&1 > /var/log/backup.log` выглядит правильно, но ошибки в лог не попадут — они уйдут на почту владельца задачи или в никуда. Правильный порядок — сначала `>`, потом `2>&1`.

</details>


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
echo 'first line' > streams-input.txt     # > создаёт файл заново, затирая прежнее содержимое
echo 'second line' >> streams-input.txt   # >> дописывает в конец, ничего не теряя
cat < streams-input.txt                   # < подаёт файл на stdin: cat даже не знает имени файла


Файл для опытов есть. Теперь запустим команду, которая одновременно и выдаёт результат, и ругается: `cat` прочитает наш файл и не найдёт `/missing-file`. Разложим два потока по разным файлам.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
cat streams-input.txt /missing-file \
  > streams-stdout.txt \
  2> streams-stderr.txt || true   # || true — чтобы ошибка не роняла всю ячейку
echo '--- stdout:'
cat streams-stdout.txt   # здесь только результат: две строки нашего файла
echo '--- stderr:'
cat streams-stderr.txt   # а здесь только жалоба на отсутствующий файл


Та же команда, но оба потока сведены в один файл: сначала stdout уходит в `streams-all.txt`, затем `2>&1` отправляет stderr туда же.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
cat streams-input.txt /missing-file > streams-all.txt 2>&1 || true   # порядок важен: сперва >, потом 2>&1
cat streams-all.txt   # в одном файле и результат, и текст ошибки


#### ❓ **Вопрос**: Команда одновременно печатает результат и сообщение об ошибке. Как сохранить их раздельно и зачем это может понадобиться?

<details>

<summary><strong>Ответ</strong></summary>

stdout направляют через `> result.txt`, stderr — через `2> errors.txt`. Результат можно обрабатывать дальше, а диагностику проверять отдельно.

</details>


#### ❓ **Вопрос**: Чем `cat file /missing-file > all.txt 2>&1` отличается от `cat file /missing-file 2>&1 > all.txt`?

<details>

<summary><strong>Ответ</strong></summary>

В первом случае stdout уже перенаправлен в файл, и `2>&1` копирует это назначение — в файле оказываются оба потока. Во втором `2>&1` выполняется раньше, когда stdout ещё связан с терминалом: в файл уйдёт только результат, а ошибка останется на экране.

</details>


## 3. Heredoc

Heredoc передаёт команде многострочный stdin. После `<<` указывается маркер окончания; `EOF` — только принятое имя. Закрывающий маркер должен стоять один на строке.

В `<<EOF` переменные раскрываются при создании текста. В `<<'EOF'` содержимое записывается буквально. Это тот же принцип, что у двойных и одинарных кавычек.

`echo -e 'one\ntwo'` подходит для короткого текста. Heredoc удобнее для большого блока: строки остаются читаемыми, не нужно повторять `echo` и экранировать кавычки.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
cat > heredoc-interpolated.txt <<EOF   # маркер без кавычек: переменные раскроются
USER=$USER
HOME=$HOME
EOF
cat heredoc-interpolated.txt   # в файле лежат уже подставленные значения


Тот же блок текста, но маркер взят в одинарные кавычки — `<<'EOF'`.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
cat > heredoc-literal.txt <<'EOF'   # маркер в кавычках: подстановка выключена целиком
USER=$USER
HOME=$HOME
EOF
cat heredoc-literal.txt   # в файле остались буквальные $USER и $HOME


#### ❓ **Вопрос**: В каком из файлов останется буквальный текст `$USER`: созданном через `<<EOF` или через `<<'EOF'`?

<details>

<summary><strong>Ответ</strong></summary>

Через `<<'EOF'`. Кавычки вокруг маркера запрещают раскрытие переменных внутри heredoc.

</details>


## 4. Переменные окружения

Полезные переменные: `$USER`, `$HOME`, `$SHELL`, `$PATH`, `$PWD`, `$OLDPWD`, `$LANG`. Полный `env` нельзя публиковать: в окружении могут быть токены.


In [ ]:
%%bash
echo "USER=$USER"   # под каким пользователем идёт работа
echo "HOME=$HOME"   # домашний каталог: именно его подставляет ~
echo "PWD=$PWD"     # текущий каталог процесса


Обычная shell-переменная видна текущей оболочке. `export` добавляет её в окружение, которое наследуют дочерние процессы.

`bash -c 'КОМАНДЫ'` запускает дочерний Bash и выполняет строку после `-c`. Одинарные кавычки оставляют `$VARIABLE` для раскрытия дочерней оболочкой.


In [ ]:
%%bash
COURSE_EXECUTION_MODE='development'                                # пока это переменная только текущей оболочки
bash -c 'echo "before export: ${COURSE_EXECUTION_MODE:-missing}"'  # дочерний процесс её не видит: missing
export COURSE_EXECUTION_MODE                                       # export переносит переменную в окружение
bash -c 'echo "after export: $COURSE_EXECUTION_MODE"'              # теперь значение унаследовано


#### ❓ **Вопрос**: Переменная `COURSE_NAME` задана без `export`. Что увидит `bash -c 'echo "${COURSE_NAME:-missing}"'` и как изменить результат?

<details>

<summary><strong>Ответ</strong></summary>

Дочерний Bash выведет `missing`. После `export COURSE_NAME` он получит значение переменной.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Через окружение программам обычно передают секреты: токен API, пароль к базе, ключ от S3 с датасетом. Это удобно (не нужно класть их в код) и опасно ровно по той же причине: любой дочерний процесс видит всё окружение целиком.

Поэтому `env`, `printenv` и `set` — команды, вывод которых нельзя не глядя вставлять в чат, в issue и в лог CI. Регулярная утечка: студент показывает «у меня не работает» скриншотом всего терминала, а в кадре `HUGGINGFACE_TOKEN=hf_...`. Токены после такого меняют.

</details>


## 5. Типы объектов, пользователи и права


### Типы объектов

Первый символ в `ls -l` обозначает тип. Флаг `-d` показывает саму директорию, а не её содержимое, поэтому `ls -ld path` удобен для проверки типа:

- `-` — обычный файл;
- `d` — директория;
- `l` — символическая ссылка;
- `c` — символьное устройство;
- `b` — блочное устройство;
- `p` — именованный канал;
- `s` — сокет.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
ln -sf streams-input.txt link-to-input   # символическая ссылка: отдельный объект, указывающий на файл
ls -ld /dev/null ~/seminar-02 link-to-input streams-input.txt   # первый символ: c, d, l и - у обычного файла


### Пользователи и группы

`whoami` показывает пользователя, `id` — UID, GID и группы, `groups` — список групп.


In [ ]:
%%bash
whoami   # имя текущего пользователя — от него зависят все проверки прав
id       # числовой UID, основная группа и все дополнительные группы
groups   # то же короче: только имена групп


`chown OWNER file` меняет владельца, `chgrp GROUP file` — группу. Смена владельца обычно требует административных прав: отдать свой файл другому пользователю без `sudo` не получится.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
touch owner-demo.txt   # свой файл в своём каталоге
chown "$USER" owner-demo.txt 2> chown-errors.txt || true   # смена владельца на самого себя разрешена
cat chown-errors.txt   # пусто: ошибок не было
ls -l owner-demo.txt   # владелец и группа — третий и четвёртый столбцы вывода


### Права

После типа идут три тройки прав: владелец, группа, остальные.

- `r = 4`: читать файл; видеть список имён директории;
- `w = 2`: изменять файл; создавать и удалять записи директории;
- `x = 1`: запускать файл; проходить по директории.

Директория не становится программой из-за права `x`: для неё этот бит разрешает поиск имени и проход по пути.

Числа складываются отдельно: `7=4+2+1`, `6=4+2`, `5=4+1`, `4=4`. Например, `640` — `rw-r-----`.

Три тройки прав и что они значат у файла и у каталога


Для директории `r` без `x` позволяет увидеть имена, но не обратиться к ним. `x` без `r` разрешает доступ к заранее известному имени. Право `x` требуется на каждой директории пути.

`chmod 640 file` задаёт права числом, `chmod u=rw,go=r file` — символически, `chmod +x file` добавляет право запуска. `stat -c '%A %a %n'` показывает символьные и числовые права.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
mkdir -p perms/shared              # каталог, на котором посмотрим права каталога
echo 'report' > perms/report.txt   # и обычный файл рядом с ним
stat -c '%A %a %n' perms/report.txt perms/shared   # права по умолчанию: их выдал umask, а не мы


Права по умолчанию щедрые: файл читают все, в каталог все могут войти. Ужмём их до «владельцу — всё, группе — чтение, остальным — ничего».


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
chmod 640 perms/report.txt   # rw- r-- ---: владельцу чтение и запись, группе только чтение
chmod 750 perms/shared       # rwx r-x ---: посторонний в каталог даже не войдёт
stat -c '%A %a %n' perms/report.txt perms/shared   # %A — символьная запись, %a — она же числом


#### ❓ **Вопрос**: Что разрешают права `--x` у бинарного файла и у директории? Можно ли при `--x` увидеть список имён внутри директории?

<details>

<summary><strong>Ответ</strong></summary>

У бинарного файла `--x` разрешает попытку запуска. У директории — проход к заранее известному имени. Список имён без `r` увидеть нельзя; скрипту для запуска интерпретатор обычно должен ещё прочитать содержимое.

</details>


#### ❓ **Вопрос**: В демке `chmod 750` выставил каталогу `rwxr-x---`. Какая цифра отвечает за группу и почему у неё нет `w`?

<details>

<summary><strong>Ответ</strong></summary>

Средняя цифра, `5`. Это `4 + 1`, то есть `r` и `x`: члены группы видят список имён и могут войти в каталог, но создавать и удалять записи в нём не могут — для этого нужен бит `w = 2`, которого в пятёрке нет.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Самый популярный способ «починить права» — `chmod 777`. Он действительно снимает симптом: после него файл доступен всем на чтение, запись и запуск. Заодно он разрешает любому пользователю машины подменить содержимое вашего скрипта, а веб-серверу — выполнить залитый через форму файл. В отчётах о взломах shared-хостингов `chmod -R 777` встречается примерно всегда.

Полезная привычка: если не работает доступ к файлу глубоко в дереве, проверять не только сам файл, но и каждый каталог по пути — праву `x` достаточно отсутствовать на одном уровне, чтобы всё сломалось, а сообщение об ошибке будет ровно то же самое.

</details>


## 6. Каталоги и файлы


### Навигация и создание

`pwd` показывает текущий каталог. `cd` меняет его, `cd ..` поднимается выше, `cd -` возвращает предыдущий. `mkdir -p` создаёт дерево каталогов. `touch` создаёт пустой файл, а для существующего файла обновляет время изменения.

Файлы с точкой в начале скрыты. `ls -a` показывает их, `ls -l` включает подробный вид, `-h` делает размеры читаемыми.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
mkdir -p files/input files/work files/result   # -p создаёт всё дерево сразу и молчит, если оно есть
cd files || exit 1                             # cd действует только внутри этой ячейки
touch input/empty.txt input/.hidden            # имя с точки в начале — скрытый файл
pwd                                            # проверяем, в каком каталоге оказались
ls -lha input                                  # -a показывает скрытые, -h — размеры в K и M


### Копирование, перемещение и удаление

`cp` копирует файл, `mv` перемещает или переименовывает. `rm` удаляет файл без корзины, `rmdir` — пустой каталог, `rm -r` — каталог с содержимым. Флаг `-f` отключает подтверждения и игнорирует отсутствующие файлы.


In [ ]:
%%bash
cd ~/seminar-02/files || exit 1
cp input/empty.txt work/             # копия появилась в work, оригинал остался в input
mv work/empty.txt work/renamed.txt   # mv и перемещает, и переименовывает — это одна операция
touch result/remove-me.txt           # файл специально для удаления
rm -- result/remove-me.txt           # -- закрывает список флагов: имя с дефисом не примут за опцию
ls -lha input work result            # смотрим, что где осталось


`rm -rf` удаляет рекурсивно и без подтверждения, поэтому ошибка в пути особенно опасна. Перед удалением проверяют `pwd`, полный раскрытый путь и содержимое каталога. Нельзя передавать туда `/`, `$HOME`, пустую переменную, непроверенный `*` или пользовательский ввод. `--` завершает список флагов: `rm -- "$name"` не примет имя файла за опцию.


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Классика жанра — строка вида `rm -rf "$STEAMROOT/"*`. Если переменная по какой-то причине оказалась пустой, путь превращается в `rm -rf /*`. Именно такая ошибка в 2015 году в скрипте запуска Steam для Linux стирала пользователям домашний каталог, а с примонтированными дисками — и их тоже. Похожие истории есть у любого крупного проекта: `rm -rf /usr` в установщике Bumblebee, «очистка кэша» в CI, снёсшая рабочую копию.

Мораль для наших ячеек: путь в `rm -rf` пишем целиком, без переменных и без `*` на конце, а перед удалением смотрим `pwd` и `ls`. Проверять переменную (`test -n "$dir"`) тоже нужно, но глазами на полный путь — надёжнее.

</details>


### История и управление строкой

Стрелки `↑` и `↓` листают историю. `Ctrl+R` ищет назад, `!!` повторяет последнюю команду. Команда `history` печатает нумерованный список прошлых команд, а `!42` повторяет команду с номером 42. `Ctrl+F` перемещает курсор вправо, а не ищет историю.

В стандартном режиме редактирования Bash `Ctrl+X` — начало сочетания, а не отдельная команда. Например, `Ctrl+X Ctrl+E` открывает текущую строку в настроенном `$EDITOR`.

Всё это работает только в интерактивной оболочке: в ячейке ноутбука и в скрипте истории нет, поэтому `history` там ничего не покажет — пробуйте в терминале.


#### ❓ **Вопрос**: Почему перед `rm -rf` недостаточно проверить только имя последнего каталога в пути?

<details>

<summary><strong>Ответ</strong></summary>

Ошибка может находиться в любой части пути или в пустой переменной. Нужно проверить полный раскрытый путь и текущий каталог.

</details>


## 7. Процессы, jobs и сигналы

Процесс — запущенный экземпляр программы. Операционная система разделяет работу на процессы, чтобы:

- **изолировать программы**: ошибка одного процесса обычно не повреждает память другого;
- **выполнять несколько задач одновременно** и распределять их между ядрами CPU;
- **учитывать и ограничивать ресурсы**: CPU, память, открытые файлы и другие объекты;
- **проверять права доступа**: у процесса есть пользователь, группы и другие параметры безопасности.

Процессы могут обмениваться данными через файлы, пайплайны и другие механизмы, но каждый имеет собственные PID, состояние, окружение, текущий каталог и набор открытых файлов. PPID указывает родительский процесс. `ps` показывает процессы системы, `jobs` — задачи текущей интерактивной оболочки.

У `ps` флаг `-p PID` выбирает процесс по PID, `-u USER` — процессы пользователя, `-o fields` — нужные поля вывода.


`&` запускает команду в фоне, `$!` содержит PID последнего фонового процесса, `wait` ожидает завершение. `kill PID` отправляет процессу SIGTERM — вежливую просьбу завершиться.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
sleep 30 > sleep.out 2> sleep.err &         # & отправляет команду в фон: shell сразу свободен
process_id=$!                               # $! — PID последнего фонового процесса
ps -o pid,ppid,stat,cmd -p "$process_id"    # STAT=S: процесс спит и ждёт своего времени
kill -TERM "$process_id"                    # SIGTERM: то же, что просто kill PID
wait "$process_id" 2>/dev/null || true      # дожидаемся конца; код 143 = 128 + сигнал 15
ps -p "$process_id" || echo 'process finished'   # процесса больше нет, ps ничего не находит


`Ctrl+C` отправляет foreground-процессу SIGINT. `Ctrl+Z` приостанавливает его через SIGTSTP. `jobs` показывает номера jobs; `bg %2` продолжает вторую в фоне, `fg %2` возвращает её на передний план.

Тем же самым сигналом можно управлять и чужим процессом по PID: SIGSTOP приостанавливает, SIGCONT продолжает. Посмотрим на колонку `STAT`: `S` — спит, `T` — остановлен.


In [ ]:
%%bash
sleep 30 &                            # новая ячейка — новый shell, поэтому запускаем процесс заново
process_id=$!
ps -o pid,stat,cmd -p "$process_id"   # STAT=S: процесс живёт обычной жизнью
kill -STOP "$process_id"              # SIGSTOP делает с процессом то же, что Ctrl+Z с текущей командой
ps -o pid,stat,cmd -p "$process_id"   # STAT=T: остановлен, времени CPU больше не получает
kill -CONT "$process_id"              # SIGCONT возвращает его в работу — это аналог bg
ps -o pid,stat,cmd -p "$process_id"   # STAT=S снова: процесс продолжил с того же места
kill -TERM "$process_id"              # убираем за собой, чтобы не оставлять мусор в системе


SIGKILL (`kill -9`) применяют только когда корректное завершение не работает: процесс не получает управление и не может ни сохранить данные, ни удалить временные файлы.


#### ❓ **Вопрос**: В `jobs` показаны `[1]` и `[2]`, обе остановлены. Как продолжить job 1 в фоне, а job 2 вернуть на передний план?

<details>

<summary><strong>Ответ</strong></summary>

Выполнить `bg %1`, затем `fg %2`.

</details>


#### ❓ **Вопрос**: В демке после `kill -STOP` колонка `STAT` показала `T`, а после `kill -CONT` — снова `S`. Какое сочетание клавиш делает с текущей командой то же, что `kill -STOP`?

<details>

<summary><strong>Ответ</strong></summary>

`Ctrl+Z`: он посылает процессу на переднем плане SIGTSTP, и тот переходит в то же состояние `T`. Вернуть его к работе можно через `bg` (продолжить в фоне) или `fg` (вернуть на передний план) — это аналог `kill -CONT`.

</details>


## 8. Пайплайны

`|` направляет stdout команды слева в stdin команды справа. stderr автоматически в пайплайн не попадает. Перед сборкой пайплайна каждую команду полезно проверить отдельно.

Обычно код пайплайна равен коду последней команды. `set -o pipefail` делает пайплайн неуспешным, если завершилась с ошибкой любая его часть.

Пайплайн: stdout одного процесса становится stdin следующего


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
echo 'pipeline data' | cat > pipeline.txt   # stdout левой команды стал stdin правой, а её вывод — файлом
cat pipeline.txt                            # всё, что прошло по трубе, осело в файле


Теперь сломаем первую команду пайплайна и посмотрим на код завершения всей конструкции.


In [ ]:
%%bash
cat /missing-file | cat        # левая команда падает, правая честно отрабатывает на пустом входе
echo "without pipefail: $?"    # код всего пайплайна — это код последней команды, то есть 0
set -o pipefail                # включаем строгий режим для пайплайнов
cat /missing-file | cat || echo 'pipeline failed'   # теперь ошибка левой части видна снаружи


#### ❓ **Вопрос**: Первая команда пайплайна завершилась с ошибкой, последняя — успешно. Как изменится код после `set -o pipefail`?

<details>

<summary><strong>Ответ</strong></summary>

Без `pipefail` берётся код последней команды. С `pipefail` весь пайплайн получит ненулевой код из-за ошибки одной из частей.

</details>


## 9. Система и ресурсы

`uname` показывает kernel и архитектуру, `/etc/os-release` — дистрибутив. `uname -a` выводит все основные сведения. `uptime` показывает время работы и load average, `free` — память, `df` — место на файловых системах. У `free -h` и `df -h` флаг `-h` делает размеры читаемыми.

Фигурные скобки группируют команды, которые выполняются последовательно в текущем Bash. Перенаправление после `}` применяется ко всей группе. После `{` нужен пробел или перенос строки, перед `}` — перенос строки или `;`.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
mkdir -p report   # каталог для собранной диагностики
{                 # всё, что напечатают команды внутри, пойдёт в один и тот же файл
  uname -a        # ядро, архитектура, имя машины
  uptime          # сколько система работает и средняя загрузка
  free -h         # оперативная память в читаемом виде
  df -h "$HOME"   # сколько места осталось на диске с домашним каталогом
} > report/system.txt 2> report/errors.txt   # одно перенаправление на всю группу команд


Ничего не напечаталось: весь вывод ушёл в файлы. Заглянем в них.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
head -n 3 report/system.txt   # первые строки отчёта: сюда попал stdout всей группы
cat report/errors.txt         # обычно пусто — значит, ни одна команда не ругалась


`ps`, `top`, `htop` показывают процессы и нагрузку. `top` и `htop` — интерактивные, в ноутбуке их запускать бесполезно (выход — `q`), а вот `ps` отлично работает в скрипте. `nvidia-smi` выводит состояние поддерживаемой NVIDIA GPU. Отсутствие `htop` или `nvidia-smi` нормально.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
ps -u "$USER" -o pid,ppid,stat,%cpu,%mem,cmd > report/processes.txt   # свои процессы — в файл
wc -l report/processes.txt   # сколько строк получилось: столько процессов плюс заголовок


In [ ]:
%%bash
which top >/dev/null && echo 'top: available' || echo 'top: not available'
which htop >/dev/null && echo 'htop: available' || echo 'htop: not available'
which nvidia-smi >/dev/null && echo 'nvidia-smi: available' || echo 'nvidia-smi: not available'


#### ❓ **Вопрос**: Нужно сохранить stdout четырёх диагностических команд в `system.txt`, а все их ошибки — в `errors.txt`. Зачем здесь удобны фигурные скобки?

<details>

<summary><strong>Ответ</strong></summary>

Одно перенаправление после `}` применяется ко всей группе; пути не нужно повторять для каждой команды.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`nvidia-smi` — первая команда, которую набирают на новом GPU-сервере, и почти единственный способ понять, что происходит с видеокартой: сколько памяти занято, какие процессы её держат и не перегрелась ли она. Типичная сцена на кластере: обучение падает с `CUDA out of memory`, `nvidia-smi` показывает чужой процесс на 20 гигабайт, и дальше вопрос решается не техникой, а переговорами.

`load average` в `uptime` — три числа: средняя длина очереди готовых к выполнению процессов за 1, 5 и 15 минут. Ориентир — число ядер: `load average` 8 на восьмиядерной машине означает полную загрузку, а на двухъядерной — что задачи стоят в очереди вчетверо дольше, чем считаются.

</details>


## 10. Первый Bash-скрипт

Последовательность команд можно записать в текстовый файл. Расширение `.sh` принято для понятности, но Bash определяет запуск не по расширению.

Первая строка `#!/usr/bin/env bash` — shebang. Она выбирает интерпретатор при прямом запуске. Строки после `#` — комментарии.

Скрипт создаётся через `<<'EOF'`, поэтому `$USER` и `$HOME` останутся внутри файла и раскроются только при его запуске.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
cat > first_script.sh <<'EOF'   # кавычки у маркера: подстановка отложена до запуска скрипта
#!/usr/bin/env bash
echo "USER=$USER"
mkdir -p "$HOME/seminar-02/script-result"
cd "$HOME/seminar-02/script-result" || exit 1
echo "Created by Bash script" > created-by-script.txt
echo "Script finished"
EOF


Файл создан, но пока это просто текст: права на запуск у него нет. Посмотрим на содержимое и добавим бит `x`.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
cat first_script.sh      # внутри лежат буквальные $USER и $HOME — они раскроются при запуске
ls -l first_script.sh    # в правах нет ни одного x: ./first_script.sh пока не запустится
chmod +x first_script.sh # добавляем право запуска
ls -l first_script.sh    # теперь x есть, и shebang определит, чем исполнять файл


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
./first_script.sh                         # запуск напрямую: нужны shebang и право x
cat script-result/created-by-script.txt   # скрипт действительно создал файл в своём каталоге


#### ❓ **Вопрос**: Какими двумя способами запустить `first_script.sh`? Для какого из них обязательны shebang и право `x`?

<details>

<summary><strong>Ответ</strong></summary>

`bash first_script.sh` запускает файл через явно выбранный интерпретатор и не требует `x` или shebang. Для `./first_script.sh` нужны право `x` и корректный shebang.

</details>


## Дополнительно

Дальше — справочник. На занятии эти разделы не показывают: они нужны, когда вы решаете задачи и вспоминаете, чем `tee` отличается от `>` и как устроена подстановка команды.


### Globs

Globs раскрывает оболочка до запуска команды:

- `*` — любое количество символов;
- `?` — один символ;
- `[0-9]` — один символ из диапазона.

Обычный `*` не выбирает скрытые файлы. Кавычки запрещают раскрытие, поэтому переменную пути заключают в кавычки, а шаблон оставляют снаружи: `"$dir"/test*.sh`.

У `ls` флаг `-t` сортирует по времени, `-r` разворачивает порядок.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
mkdir -p globs
touch globs/test-one.sh globs/test-two.sh globs/.test-hidden.sh   # третий файл — скрытый

ls -lrth globs/test*.sh   # шаблон не поймал скрытый файл: обычный * его не видит
ls -lrtha globs           # -a показывает всё, включая .test-hidden.sh


### `head`, `tee` и `wc`

`head -n 5` оставляет первые пять строк. `tee file` сохраняет stdin в файл и одновременно передаёт его дальше; `tee -a file` дописывает. `wc -l` считает строки, `wc -w` — слова, `wc -c` — байты.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
cat /etc/os-release \
  | tee os-release-copy.txt \
  | head -n 5 \
  | wc -l          # на экране только число строк, а полная копия осталась в файле


### Копирование каталогов

`cp -r` рекурсивно копирует каталог. Запись `source/.` означает всё содержимое `source`, включая скрытые имена, без дополнительного уровня `source`.


In [ ]:
%%bash
cd ~/seminar-02 || exit 1
mkdir -p copy-source copy-destination
touch copy-source/visible.txt copy-source/.hidden.txt   # один обычный файл и один скрытый

cp -r copy-source/. copy-destination/   # «/.» переносит содержимое, а не сам каталог
ls -la copy-destination                 # скрытый файл тоже на месте


### Параметры скрипта

`$0` содержит имя скрипта, `$1`, `$2` и далее — переданные аргументы, `$#` — их количество. `exit N` завершает скрипт с кодом `N`.

```bash
#!/usr/bin/env bash
echo "script=$0 arguments=$#"
echo "input=$1 label=$2"
exit 0
```

При запуске `./report.sh data.csv train` значениями `$1` и `$2` будут `data.csv` и `train`.


### Подстановка команды

`$(command)` запускает команду и подставляет её stdout в текущую строку.

```bash
current_user=$(whoami)
echo "user=$current_user"
```


### Проверки

`test "$left" -eq "$right"` сравнивает числа, `test "$left" = "$right"` — строки. `test -f path` проверяет обычный файл, `test -d path` — директорию. Команда возвращает `0` при успехе и ненулевой код при провале.

```bash
test "$(wc -c < first.txt)" -eq "$(wc -c < second.txt)"
```


### Фоновые процессы

`$!` содержит PID последнего запущенного в фоне процесса. `wait PID` ожидает именно этот процесс и возвращает его код завершения.

```bash
sleep 2 &
process_id=$!
wait "$process_id"
wait_status=$?
echo "exit_code=$wait_status"
```
